# 06 – EDA results and analytical dataset

## A notebook célja

A notebook lezárja a Balaton vízállás-adatainak feltáró elemzését. Összefoglalja az 01–05. notebookok eredményeit, rögzíti az elemzési korlátokat, majd előállít két további elemzésre alkalmas állományt:

- **mérési szintű adatkészlet**: az eredeti vízállásrekordok megőrzésével, minőségi jelölőkkel;
- **napi szintű adatkészlet**: napi átlag, minimum, maximum, terjedelem és mérési darabszám.

A notebook nem készít előrejelző modellt, nem végez további idősormodellezést, és nem von be külső magyarázó adatokat.

## 1. Könyvtárak és projektútvonal

A projekt gyökérkönyvtárának meghatározása akkor is működik, ha a Jupyter a projekt gyökeréből vagy a `notebooks` mappából indul.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT_ROOT = next((p for p in candidates if (p / "data").exists() and (p / "notebooks").exists()), Path.cwd().parent)
DATA_FILE = PROJECT_ROOT / "data" / "processed" / "balaton_2016-2026_vizallas_teljes.csv"
REPORTS_DIR = PROJECT_ROOT / "reports"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

print("Projekt gyökér:", PROJECT_ROOT)
print("Bemeneti adat:", DATA_FILE)

## 2. Adatok betöltése és alapellenőrzés

A bemeneti CSV pontosvesszővel tagolt, UTF-8 BOM kódolású. A mérési szintű adatállományt nem aggregáljuk és nem töröljük belőle automatikusan a duplikált időpontokat.

In [ ]:
df = pd.read_csv(DATA_FILE, sep=";", encoding="utf-8-sig")
df["idopont"] = pd.to_datetime(df["idopont"], errors="coerce")
df["vizallas_cm"] = pd.to_numeric(df["vizallas_cm"], errors="coerce")
df = df.sort_values(["idopont", "forras_fajl", "forras_rekord_sorszama"]).reset_index(drop=True)

print("Sorok, oszlopok:", df.shape)
display(df.head())
display(df.dtypes)

In [ ]:
quality = pd.DataFrame({
    "mutato": [
        "Rekordok száma",
        "Hiányzó időpont",
        "Hiányzó vízállás",
        "Duplikált időpontú rekordok",
        "Egyedi időpontok száma",
        "Nem növekvő időpontok az időrendbe rendezés előtt"
    ],
    "ertek": [
        len(df),
        int(df["idopont"].isna().sum()),
        int(df["vizallas_cm"].isna().sum()),
        int(df.duplicated("idopont", keep=False).sum()),
        int(df["idopont"].nunique()),
        int((pd.to_datetime(pd.read_csv(DATA_FILE, sep=";", encoding="utf-8-sig")["idopont"], errors="coerce").diff().dropna() < pd.Timedelta(0)).sum())
    ]
})
display(quality)
quality.to_csv(REPORTS_DIR / "06_adatminosegi_ellenorzes.csv", index=False, encoding="utf-8-sig", sep=";")

## 3. Duplikált időpontok részletes vizsgálata

A duplikált időpont önmagában nem jelenti azt, hogy a rekord hibás. Külön megvizsgáljuk, hogy azonos időponthoz azonos vagy eltérő vízállásértékek tartoznak-e.

In [ ]:
dup = df[df.duplicated("idopont", keep=False)].copy()
dup_groups = (dup.groupby("idopont")
              .agg(rekordok=("vizallas_cm", "size"),
                   kulonbozo_vizallas=("vizallas_cm", "nunique"),
                   minimum=("vizallas_cm", "min"),
                   maximum=("vizallas_cm", "max"))
              .reset_index())
dup_groups["ellentmondasos_vizallas"] = dup_groups["kulonbozo_vizallas"] > 1

print("Duplikált időpontcsoportok:", len(dup_groups))
print("Eltérő vízállásértékű duplikált időpontcsoportok:",
      int(dup_groups["ellentmondasos_vizallas"].sum()))
display(dup_groups.head(10))
dup_groups.to_csv(REPORTS_DIR / "06_duplikalt_idopontok_vizsgalata.csv",
                  index=False, encoding="utf-8-sig", sep=";")

## 4. Az EDA eredményeinek összefoglalása

A következő táblák az 05. notebook által előállított jelentésekből készülnek. A teljes naptári éveket külön kell kezelni a részleges 2016-os és 2026-os évektől.

In [ ]:
def read_report(filename):
    path = REPORTS_DIR / filename
    if not path.exists():
        raise FileNotFoundError(f"Hiányzó jelentés: {path}")

    # A jelentések vesszővel vagy pontosvesszővel tagoltak lehetnek.
    # A sep=None automatikusan felismeri az elválasztójelet.
    report = pd.read_csv(
        path,
        sep=None,
        engine="python",
        encoding="utf-8-sig"
    )
    report.columns = report.columns.str.strip()
    return report

annual = read_report("05_eves_vizallas_statisztikak.csv")
frequency = read_report("05_eves_meresi_gyakorisag.csv")
seasonal = read_report("05_evszakos_vizallas_statisztikak.csv")
monthly = read_report("05_havi_vizallas_statisztikak.csv")
extremes = read_report("05_eves_minimum_maximum_datumok_teljes_evek.csv")

display(annual)
display(frequency)
display(seasonal)

### 4.1 Mérési gyakoriság változása

A mérési időköz mediánja 2016–2021 között 30 perc, 2022-től 15 perc. A napi 96 mérés a 15 perces mérési rendnek felel meg. A mérési gyakoriság változása miatt az éves nyers rekorddarabszámok közvetlen összehasonlítása félrevezető lehet.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(frequency["ev"], frequency["napi_meresszam_median"], marker="o")
plt.axvline(2021.5, linestyle="--")
plt.title("Éves napi mérési darabszám mediánja")
plt.xlabel("Év")
plt.ylabel("Mérés / nap")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### 4.2 Havi és évszakos mintázatok

A havi és évszakos átlagok leíró összehasonlítások. Az eltérések értelmezéséhez figyelembe kell venni az időbeli függőséget, a mérési gyakoriság változását és az egyes évek közötti különbségeket.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(monthly["honap_nev"], monthly["atlag"], marker="o")
plt.title("Átlagos vízállás hónapok szerint")
plt.xlabel("Hónap")
plt.ylabel("Átlagos vízállás (cm)")
plt.xticks(range(1, 13))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(monthly)
display(seasonal)

## 5. Az elemzési korlátok rögzítése

- 2016 és 2026 részleges év; éves összehasonlításnál ezt jelezni kell.
- 2022-től a mérési gyakoriság megváltozott, ezért a nyers rekordok darabszáma nem használható önmagában az évek összevetésére.
- A duplikált időpontokat nem töröljük automatikusan.
- A vízállás változása önmagában nem azonos a párolgással vagy a tó víztérfogat-változásával.
- Külső magyarázó változók (például csapadék, szél, léghőmérséklet és vízügyi beavatkozások) ebben a projektben nem szerepelnek.
- A notebook nem végez előrejelzést vagy további idősormodellezést.

## 6. Mérési szintű elemzési adatkészlet

A forrásrekordokat megtartjuk. Hozzáadunk néhány, későbbi feldolgozást segítő oszlopot: naptári év, hónap, nap, mérési időpont, duplikált időpont jelölése és a mérési gyakoriság szerinti időszak.

In [ ]:
measurement = df.copy()
measurement["ev"] = measurement["idopont"].dt.year
measurement["honap"] = measurement["idopont"].dt.month
measurement["nap"] = measurement["idopont"].dt.date.astype("string")
measurement["ora_perc"] = measurement["idopont"].dt.strftime("%H:%M")
measurement["duplikalt_idopont"] = measurement.duplicated("idopont", keep=False)
measurement["meresi_rend_idoszak"] = np.where(
    measurement["idopont"] < pd.Timestamp("2022-01-01"),
    "2021-ig: jellemzően 30 perc",
    "2022-től: jellemzően 15 perc"
)
measurement_file = PROCESSED_DIR / "balaton_vizallas_meresi_szintu_elemzesi_adatok.csv"
measurement.to_csv(measurement_file, index=False, encoding="utf-8-sig", sep=";")
print("Mentve:", measurement_file)
display(measurement.head())

## 7. Napi szintű elemzési adatkészlet

A napi összesítés a mérési szintű állományból készül. A napi mérési darabszámot is megtartjuk, így látható, hogy az adott napi statisztika hány mérésen alapul.

In [ ]:
daily = (measurement.dropna(subset=["idopont", "vizallas_cm"])
         .assign(nap_dt=lambda x: x["idopont"].dt.floor("D"))
         .groupby("nap_dt")
         .agg(
             napi_atlag_cm=("vizallas_cm", "mean"),
             napi_median_cm=("vizallas_cm", "median"),
             napi_min_cm=("vizallas_cm", "min"),
             napi_max_cm=("vizallas_cm", "max"),
             meresek_szama=("vizallas_cm", "count"),
             duplikalt_idopontu_rekordok=("duplikalt_idopont", "sum")
         )
         .reset_index()
         .rename(columns={"nap_dt": "datum"}))
daily["napi_terjedelem_cm"] = daily["napi_max_cm"] - daily["napi_min_cm"]
daily["ev"] = daily["datum"].dt.year
daily["honap"] = daily["datum"].dt.month
daily["evszak"] = daily["honap"].map({
    12: "Tél", 1: "Tél", 2: "Tél",
    3: "Tavasz", 4: "Tavasz", 5: "Tavasz",
    6: "Nyár", 7: "Nyár", 8: "Nyár",
    9: "Ősz", 10: "Ősz", 11: "Ősz"
})
daily_file = PROCESSED_DIR / "balaton_vizallas_napi_elemzesi_adatok.csv"
daily.to_csv(daily_file, index=False, encoding="utf-8-sig", sep=";")
print("Mentve:", daily_file)
display(daily.head())

## 8. Kimeneti állományok ellenőrzése

A két adatkészlet rekordszámát és időbeli lefedettségét ellenőrizzük.

In [ ]:
check = pd.DataFrame([
    {
        "adatkeszlet": "Mérési szintű",
        "rekordok": len(measurement),
        "elso_datum": measurement["idopont"].min(),
        "utolso_datum": measurement["idopont"].max()
    },
    {
        "adatkeszlet": "Napi szintű",
        "rekordok": len(daily),
        "elso_datum": daily["datum"].min(),
        "utolso_datum": daily["datum"].max()
    }
])
display(check)
check.to_csv(REPORTS_DIR / "06_elemzesi_adatkimenetek_ellenorzese.csv",
             index=False, encoding="utf-8-sig", sep=";")

## 9. Projektzárás és lehetséges folytatás

A GitHub-projekt lezárásakor a fő eredmény egy dokumentált, ellenőrzött vízállás-adatkészlet és annak napi aggregált változata. A további kutatási irányok külön projektként kezelhetők:

- hosszabb távú trendek és szezonális komponensek vizsgálata;
- csapadék-, szél- és léghőmérséklet-adatok bevonása;
- vízügyi beavatkozások, különösen a Sió-zsilip működésének figyelembevétele;
- későbbi előrejelzési feladatok kialakítása és referencia-modellek összehasonlítása.

Ezek nem részei ennek a projektnek és ebben a notebookban nem készül hozzájuk modell.